In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Generating Random Permutations

In [ ]:
function randomInt(minIncl: number, maxIncl: number): number {
  return Math.floor(Math.random() * (maxIncl - minIncl + 1)) + minIncl;
}

In [ ]:
function permute<T>(L: T[]): T[] {
  if (L.length <= 1) return [...L];
  const k = randomInt(0, L.length - 1);
  const left = L.slice(0, k);
  const right = L.slice(k + 1);
  return [...permute(left), ...right, L[k]];
}

In [ ]:
for (let i = 0; i < 20; i++) {
  console.log(permute([1, 2, 3, 4, 5]));
}

# Case Study: Computation of Probabilities in Poker 

This notebook shows how to compute probabilities in 
<A href="https://en.wikipedia.org/wiki/Texas_hold_%27em">Texas Hold'em Poker</A>.  Texas Hold'em is played with a deck of 52 cards.  The cards have 13 different *values* and four different *suits*.  The values are:

In [ ]:
const Values = ["2", "3", "4", "5", "6", "7", "8", "9", "T",
                "J", "Q", "K", "A"];

The four *suits* are *clubs*, *hearts*, *diamonds*, and *spades*.  We represent these suits by their first letter:

In [ ]:
const Suits = ["c", "h", "d", "s"];

Now the *deck* of cards is convieniently represented by the Cartesian product of the two sets <tt>Values</tt> and <tt>Suits</tt>.

In [ ]:
const Deck: [string, string][] = [];
for (const v of Values) {
  for (const s of Suits) {
    Deck.push([v, s]);
  }
}

Let's assume you have been dealt the three of clubs and the three of spades.  In Texas Hold'em, the hand dealt to a player at the beginning is called the *hole*, so we have:

In [ ]:
const Hole: [string, string][] = [
  ["3", "c"],
  ["3", "s"],
];

The remaining cards are all other cards, lets call them <tt>rest</tt>:

In [ ]:
const Rest = Deck.filter(
  (card) => !Hole.some((h) => h[0] === card[0] && h[1] === card[1])
);

In [ ]:
const combinations = (50 * 49 * 48 * 47 * 46) / (1 * 2 * 3 * 4 * 5);
console.log(combinations);

We are interested in the probability that on the *river* you will improve your hand to *four of a kind*.  As there are 5 cards that will be dealt,
three cards in the *flop*, the *turn*, and the *river*, there are 
$$ (50 \cdot 49 \cdot 48 \cdot 47 \cdot 46) / (1 \cdot 2 \cdot 3 \cdot 4 \cdot 5) = 2\,118\,760 $$
possibilities for these cards.  Let's pretend that we do not have enough main memory to store all these different possibilities.  Instead, we want to run a Monte-Carlo Simulation.

The argument `n` is the number of times we run the experiment and `Rest` is the set of cards that are still unaccounted for.

In [ ]:
function getFourOfAKind(n: number, rest: [string, string][]): number {
  let numFourOfAKind = 0;
  let restArray = [...rest];

  for (let i = 0; i < n; i++) {
    restArray = permute(restArray);
    const Cards = new Set<[string, string]>([
      ...Hole,
      ...restArray.slice(0, 5),
    ]);

    const count3 = Array.from(Cards).filter((card) => card[0] === "3").length;
    if (count3 === 4) numFourOfAKind++;
  }

  return numFourOfAKind / n;
}

In [ ]:
console.time("Four of a kind simulation");
const probability = getFourOfAKind(1_000_000, Rest);
console.timeEnd("Four of a kind simulation");
console.log(probability);